# Buổi 15 — Lab

Chạy từng ô từ trên xuống. Mỗi bước ứng với một mục trong tài liệu (mục 5). Bạn sửa `backtest.py`; các ô tự dùng bản mới.

In [ ]:
# sửa tệp .py trong code/ thì các ô sau tự dùng bản mới, không cần khởi động lại
%load_ext autoreload
%autoreload 2

## Bước 1 — Ba cách ước lượng sai số (mục 4.1)

Ô này chạy khoảng nửa phút (rừng ngẫu nhiên học lại nhiều lần).

In [ ]:
%matplotlib inline
import warnings

import backtest as bt
import numpy as np
import pandas as pd

warnings.simplefilter("ignore")
y = bt.doc_dien()
print(len(y), "giờ,", y.index.min(), "→", y.index.max())
bang = bt.so_sanh_cach_chia(y)
print(bang.round(1).to_string(index=False))

## Bước 2 — Bộ backtest: cửa sổ, tầm h, đối chiếu statsforecast (mục 4.2, 4.3)

Không cần sửa.

In [ ]:
nho = pd.DataFrame({"unique_id": "a", "ds": np.arange(20), "y": np.arange(20.0)})
for cs in bt.chia_cua_so(nho, h=3, so_cua_so=2, buoc=2, gap=1):
    print(cs)
m4 = bt.doc_m4_gio()
df = bt.m4_dang_dai(m4)
print("số chuỗi dài nhất:", df["unique_id"].nunique(), "| lệch statsforecast:", bt.doi_chieu_statsforecast(df))
kq = bt.backtest(df, bt.seasonal_naive, h=bt.H_M4, so_cua_so=3)
print(kq.head())
print(kq.groupby("cutoff").size())

## Bước 3 — Ước lượng bằng rolling origin (mục 4.1, 4.2)

Sửa `uoc_luong_sai_so` rồi chạy lại ô này.

In [ ]:
bang = bt.so_sanh_cach_chia(y)
print(bang.round(1).to_string(index=False))
rs = bt.rung_so_voi_snaive(y)
print(rs["theo cửa sổ"].describe().round(0))
rs["theo cửa sổ"].plot(marker="o", figsize=(8, 3), ylabel="MAE một cửa sổ (MW)");

## Bước 4 — Tune, chọn, báo cáo trên ba đoạn riêng (mục 4.4)

Sửa `chon_va_bao_cao` rồi chạy lại ô này.

In [ ]:
dung = bt.chon_va_bao_cao(m4)
cot = [c for c in ("MASE lúc chọn (A)", "MASE báo cáo (B)", "seasonal naive 24 (B)") if c in dung]
print(dung[cot].median().round(3))
print(dung["chọn"].value_counts())
sai = bt.chon_va_bao_cao_cung_cua_so(m4)
print("chọn và báo cáo cùng đoạn:", round(float(sai["MASE báo cáo (B)"].median()), 3))

## Bước 5 — Diebold–Mariano (mục 4.5)

Sửa `diebold_mariano` rồi chạy lại ô này. Cuối cùng: `python lab.py check` trong terminal.

In [ ]:
kiem = y[y.index >= bt.MOC_HOLD_OUT].index
truoc = y[(y.index < bt.MOC_HOLD_OUT) & (y.index >= bt.MOC_HOLD_OUT - pd.Timedelta("91D"))].index
print("w tune trên hold-out:", bt.tune_w(y, kiem), "| w tune trên 3 tháng trước mốc:", bt.tune_w(y, truoc))
kq = bt.so_sanh_tron_voi_snaive(y, 0.8)
print({k: v for k, v in kq.items() if not k.startswith("e_")})
k = rs["kq"]
print("rừng so với seasonal naive trên 28 cửa sổ:",
      bt.diebold_mariano(k["y"] - k["du_bao"], k["y"] - k["seasonal_naive"], h=bt.H_DIEN))